# Unbounded Graph Expansion — Cospel/rbm-ae-tf

**Smell (`au.py` lines 25, 28, 55):** Inside the training loop, `tf.Variable` objects are created on every iteration — weight matrices `W`, biases `bh`, `bv` are re-instantiated each pass. In TF1/eager mode, this creates new graph nodes every loop iteration, growing the graph unboundedly and consuming ever-increasing memory and compute.

**Fix:** Create all `tf.Variable` objects **once** before the loop, then reuse them across iterations. Wrap the update logic in `@tf.function` so the computation is traced once and reused.

In [ ]:
!pip install -q codecarbon

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import mnist
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS    = 10
N_EPOCHS  = 3
BATCH     = 64
N_VISIBLE = 784
N_HIDDEN  = 128
LR        = 0.01
K_CD      = 1    # contrastive divergence steps

(x_tr, _), (x_te, _) = mnist.load_data()
x_tr = (x_tr.reshape(-1, 784).astype('float32') / 255.0)
x_te = (x_te.reshape(-1, 784).astype('float32') / 255.0)
# Binarize
x_tr = (x_tr > 0.5).astype('float32')
x_te = (x_te > 0.5).astype('float32')

n_batches = len(x_tr) // BATCH
print(f'MNIST binarized: train={x_tr.shape}  batches/epoch={n_batches}')

## BEFORE — Smell Active (tf.Variable created inside loop — graph grows unboundedly)

In [ ]:
results_before = []

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()

    tracker = EmissionsTracker(
        project_name=f'RBM_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    total_loss = 0.0
    step_count = 0

    for epoch in range(N_EPOCHS):
        idx = np.random.permutation(len(x_tr))
        for b in range(n_batches):
            batch = x_tr[idx[b*BATCH:(b+1)*BATCH]]

            # ❌ SMELL (mirrors au.py lines 25, 28, 55):
            # tf.Variable created INSIDE the loop — new graph nodes each iteration
            W  = tf.Variable(tf.random.normal([N_VISIBLE, N_HIDDEN], stddev=0.01))  # ← smell
            bh = tf.Variable(tf.zeros([N_HIDDEN]))                                   # ← smell
            bv = tf.Variable(tf.zeros([N_VISIBLE]))                                  # ← smell

            v0 = tf.constant(batch)
            h0_prob = tf.nn.sigmoid(tf.matmul(v0, W) + bh)
            h0 = tf.cast(tf.random.uniform(tf.shape(h0_prob)) < h0_prob, tf.float32)
            v1_prob = tf.nn.sigmoid(tf.matmul(h0, tf.transpose(W)) + bv)
            v1 = tf.cast(tf.random.uniform(tf.shape(v1_prob)) < v1_prob, tf.float32)
            h1_prob = tf.nn.sigmoid(tf.matmul(v1, W) + bh)

            pos = tf.matmul(tf.transpose(v0), h0_prob)
            neg = tf.matmul(tf.transpose(v1), h1_prob)
            W.assign_add(LR * (pos - neg) / BATCH)
            bh.assign_add(LR * tf.reduce_mean(h0_prob - h1_prob, axis=0))
            bv.assign_add(LR * tf.reduce_mean(v0 - v1_prob, axis=0))

            recon_err = tf.reduce_mean(tf.square(v0 - v1_prob)).numpy()
            total_loss += recon_err
            step_count += 1

        print(f'  epoch {epoch+1}/{N_EPOCHS}  recon_err={total_loss/step_count:.4f}')

    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run, 'avg_recon_err': round(total_loss / step_count, 6),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    gc.collect()

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/Cospel_rbm_ae_tf_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/Cospel_rbm_ae_tf_before.csv')

## AFTER — Smell Fixed (tf.Variable created once before loop, @tf.function step)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()

    # ✅ FIX: tf.Variable created ONCE outside the loop
    W  = tf.Variable(tf.random.normal([N_VISIBLE, N_HIDDEN], stddev=0.01))
    bh = tf.Variable(tf.zeros([N_HIDDEN]))
    bv = tf.Variable(tf.zeros([N_VISIBLE]))

    # ✅ FIX: @tf.function compiles the step once — no new graph nodes per call
    @tf.function
    def cd_step(batch_v):
        v0 = batch_v
        h0_prob = tf.nn.sigmoid(tf.matmul(v0, W) + bh)
        h0 = tf.cast(tf.random.uniform(tf.shape(h0_prob)) < h0_prob, tf.float32)
        v1_prob = tf.nn.sigmoid(tf.matmul(h0, tf.transpose(W)) + bv)
        v1 = tf.cast(tf.random.uniform(tf.shape(v1_prob)) < v1_prob, tf.float32)
        h1_prob = tf.nn.sigmoid(tf.matmul(v1, W) + bh)
        pos = tf.matmul(tf.transpose(v0), h0_prob)
        neg = tf.matmul(tf.transpose(v1), h1_prob)
        W.assign_add(LR * (pos - neg) / tf.cast(tf.shape(v0)[0], tf.float32))
        bh.assign_add(LR * tf.reduce_mean(h0_prob - h1_prob, axis=0))
        bv.assign_add(LR * tf.reduce_mean(v0 - v1_prob, axis=0))
        return tf.reduce_mean(tf.square(v0 - v1_prob))

    tracker = EmissionsTracker(
        project_name=f'RBM_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    total_loss = 0.0
    step_count = 0

    for epoch in range(N_EPOCHS):
        idx = np.random.permutation(len(x_tr))
        for b in range(n_batches):
            batch = tf.constant(x_tr[idx[b*BATCH:(b+1)*BATCH]])
            recon_err = cd_step(batch).numpy()
            total_loss += recon_err
            step_count += 1
        print(f'  epoch {epoch+1}/{N_EPOCHS}  recon_err={total_loss/step_count:.4f}')

    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run, 'avg_recon_err': round(total_loss / step_count, 6),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del W, bh, bv, cd_step; gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/Cospel_rbm_ae_tf_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/Cospel_rbm_ae_tf_after.csv')

In [ ]:
print('\n=== SUMMARY: Cospel/rbm-ae-tf ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")